In [ ]:
import geopandas as gpd
import numpy as np
from shapely.geometry import LineString, Point, MultiPolygon
from shapely.ops import split, unary_union
import pandas as pd
import os
import logging
import time
from datetime import timedelta

# =====================================================
# CONFIGURATION
# =====================================================
ROOT_DIR = r"C:\Users\KyleSteen.AzureAD\Documents\Splitting_ROW_every_100m"

ROW_GPKG = os.path.join(
    ROOT_DIR,
    "Final_ROWs_Delaware.gpkg"
)
HW_GPKG = os.path.join(ROOT_DIR, "NHS_Delaware.gpkg")

# Outputs
TRANSECT_LINES_GPKG   = os.path.join(ROOT_DIR, "CONUS_transect_lines.gpkg")
CLIPPED_LINES_GPKG    = os.path.join(ROOT_DIR, "CONUS_clipped_lines.gpkg")
SPLIT_POLYGONS_GPKG   = os.path.join(ROOT_DIR, "CONUS_split_polygons.gpkg")

TRANSECT_HALF_LENGTH = 300   # meters each side
INTERVAL             = 250    # meters between transects

# =====================================================
# LOGGER
# =====================================================
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger()

def eta(start, done, total):
    if done == 0:
        return "estimating..."
    rate = done / (time.time() - start)
    return str(timedelta(seconds=int((total - done) / rate)))

start_time = time.time()
logger.info("Starting ROW polygon splitting workflow...")

# =====================================================
# LOAD DATA
# =====================================================
logger.info("Loading ROW polygons...")
rows = gpd.read_file(ROW_GPKG)
logger.info(f"  Loaded {len(rows):,} ROW polygons | CRS: {rows.crs}")

logger.info("Loading highway centerlines...")
hw_lines = gpd.read_file(HW_GPKG)

if hw_lines.crs != rows.crs:
    logger.info(f"  Reprojecting highways from {hw_lines.crs} → {rows.crs}")
    hw_lines = hw_lines.to_crs(rows.crs)

logger.info(f"  Loaded {len(hw_lines):,} highway segments")

# =====================================================
# STEP 1 — GENERATE PERPENDICULAR TRANSECTS (every 100m)
# =====================================================
logger.info("Generating perpendicular transects every 100m...")
transect_geoms = []

hw_start = time.time()
total_hw = len(hw_lines)

for i, hw_row in enumerate(hw_lines.itertuples(), start=1):
    line = hw_row.geometry
    if line is None or line.is_empty:
        continue

    # Sample at 0, 100, 200, ... up to line.length
    distances = np.arange(0, line.length + INTERVAL, INTERVAL)

    for dist in distances:
        dist = min(dist, line.length)
        point = line.interpolate(dist)

        # Tangent vector via tiny offset
        d0 = max(dist - 1e-4, 0)
        d1 = min(dist + 1e-4, line.length)
        tang_start = line.interpolate(d0)
        tang_end   = line.interpolate(d1)

        dx = tang_end.x - tang_start.x
        dy = tang_end.y - tang_start.y

        # Perpendicular (rotate 90°)
        perp_dx = -dy
        perp_dy =  dx
        norm = np.hypot(perp_dx, perp_dy)
        if norm == 0:
            continue
        perp_dx /= norm
        perp_dy /= norm

        p1 = Point(point.x - perp_dx * TRANSECT_HALF_LENGTH,
                   point.y - perp_dy * TRANSECT_HALF_LENGTH)
        p2 = Point(point.x + perp_dx * TRANSECT_HALF_LENGTH,
                   point.y + perp_dy * TRANSECT_HALF_LENGTH)

        transect_geoms.append(LineString([p1, p2]))

    if i % 500 == 0 or i == total_hw:
        logger.info(
            f"  Highways {i:,}/{total_hw:,} ({i/total_hw:.1%}) "
            f"| Transects so far: {len(transect_geoms):,} "
            f"| ETA {eta(hw_start, i, total_hw)}"
        )

logger.info(f"Total transects generated: {len(transect_geoms):,}")

transect_gdf = gpd.GeoDataFrame(
    {"transect_id": range(len(transect_geoms))},
    geometry=transect_geoms,
    crs=rows.crs
)

logger.info(f"Writing transect lines → {TRANSECT_LINES_GPKG}")
transect_gdf.to_file(TRANSECT_LINES_GPKG, driver="GPKG")

# =====================================================
# STEP 2 — CLIP TRANSECTS TO ROW POLYGONS
# =====================================================
logger.info("Clipping transects to ROW polygons (spatial join + intersection)...")

clip_start = time.time()

# Spatial join for candidate pairs (R-tree bbox pre-filter)
candidates = gpd.sjoin(
    transect_gdf,
    rows[["geometry"]].reset_index(names="row_idx"),
    how="inner",
    predicate="intersects"
)

logger.info(f"  Candidate pairs from spatial join: {len(candidates):,}")

# Pre-index ROW geometries for fast lookup
row_geom_lookup = rows["geometry"]

# Vectorised-ish intersection (apply per candidate pair)
clipped_geoms = []
clipped_ids   = []

clip_iter_start = time.time()
total_cands = len(candidates)

for n, (idx, cand_row) in enumerate(candidates.iterrows(), start=1):
    row_geom = row_geom_lookup.iloc[cand_row["row_idx"]]
    clipped  = cand_row.geometry.intersection(row_geom)

    if clipped.is_empty:
        continue

    clipped_geoms.append(clipped)
    clipped_ids.append(cand_row["transect_id"])

    if n % 50_000 == 0 or n == total_cands:
        logger.info(
            f"  Intersections {n:,}/{total_cands:,} ({n/total_cands:.1%}) "
            f"| ETA {eta(clip_iter_start, n, total_cands)}"
        )

clipped_gdf = gpd.GeoDataFrame(
    {"transect_id": clipped_ids},
    geometry=clipped_geoms,
    crs=rows.crs
)

# Keep only line/multiline results (drop point-touches)
clipped_gdf = clipped_gdf[
    clipped_gdf.geometry.geom_type.isin(["LineString", "MultiLineString"])
].reset_index(drop=True)

logger.info(f"  Clipped lines retained: {len(clipped_gdf):,}")
logger.info(f"Writing clipped lines → {CLIPPED_LINES_GPKG}")
clipped_gdf.to_file(CLIPPED_LINES_GPKG, driver="GPKG")

# =====================================================
# STEP 3 — SPLIT ROW POLYGONS USING TRANSECTS
# =====================================================
logger.info("Splitting ROW polygons using transect lines...")

split_start = time.time()
split_polygons = []

# Spatial join to find which transects intersect which ROW polygon
transects_on_rows = gpd.sjoin(
    clipped_gdf[["geometry"]],
    rows[["geometry"]].reset_index(names="row_idx"),
    how="inner",
    predicate="intersects"
)

# Group clipped transects by the ROW polygon they touch
grouped = transects_on_rows.groupby("row_idx")
total_groups = len(grouped)

for g, (row_idx, group) in enumerate(grouped, start=1):
    row_poly = rows.geometry.iloc[row_idx]

    # Union all transect lines that touch this ROW polygon
    cutter = unary_union(group.geometry.values)

    try:
        # Buffer the cutter line by a tiny amount to ensure clean splitting
        # then use difference-based split: subtract thin buffered strips and
        # collect the resulting polygon fragments
        #
        # Shapely's split() requires a single splitter; for multiple lines
        # we iterate and progressively split the remaining pieces.
        pieces = [row_poly]

        for line in (group.geometry.values
                     if hasattr(group.geometry.values[0], 'geoms') is False
                     else list(cutter.geoms)):
            new_pieces = []
            splitter = line.buffer(1e-6)   # tiny buffer → makes a thin polygon cutter
            for piece in pieces:
                try:
                    diff = piece.difference(splitter)
                    if diff.is_empty:
                        continue
                    if diff.geom_type == "MultiPolygon":
                        new_pieces.extend(list(diff.geoms))
                    elif diff.geom_type == "Polygon":
                        new_pieces.append(diff)
                    else:
                        new_pieces.append(piece)   # fallback: keep original
                except Exception:
                    new_pieces.append(piece)
            pieces = new_pieces if new_pieces else pieces

        split_polygons.extend(pieces)

    except Exception as e:
        logger.warning(f"  Could not split ROW index {row_idx}: {e}")
        split_polygons.append(row_poly)   # keep original if split fails

    if g % 1000 == 0 or g == total_groups:
        logger.info(
            f"  ROW polygons processed {g:,}/{total_groups:,} ({g/total_groups:.1%}) "
            f"| Split pieces so far: {len(split_polygons):,} "
            f"| ETA {eta(split_start, g, total_groups)}"
        )

split_gdf = gpd.GeoDataFrame(
    {"split_id": range(len(split_polygons))},
    geometry=split_polygons,
    crs=rows.crs
)

# Drop any degenerate geometries
split_gdf = split_gdf[
    split_gdf.geometry.is_valid &
    ~split_gdf.geometry.is_empty &
    split_gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].reset_index(drop=True)

logger.info(f"  Final split polygon count: {len(split_gdf):,}")
logger.info(f"Writing split polygons → {SPLIT_POLYGONS_GPKG}")
split_gdf.to_file(SPLIT_POLYGONS_GPKG, driver="GPKG")

# =====================================================
# DONE
# =====================================================
logger.info(
    f"Workflow complete in {timedelta(seconds=int(time.time() - start_time))}"
)
logger.info("Outputs:")
logger.info(f"  Transect lines  → {TRANSECT_LINES_GPKG}")
logger.info(f"  Clipped lines   → {CLIPPED_LINES_GPKG}")
logger.info(f"  Split polygons  → {SPLIT_POLYGONS_GPKG}")

In [ ]:
##########

In [ ]:
##### Centerline Generation Script 

import geopandas as gpd
import numpy as np
import os
import logging
import time
from datetime import timedelta
from shapely.geometry import MultiLineString, LineString
from shapely.ops import unary_union, linemerge
from centerline.geometry import Centerline

# =====================================================
# CONFIGURATION
# =====================================================
ROOT_DIR = r"C:\Users\KyleSteen.AzureAD\Documents\Splitting_ROW_every_100m"

ROW_GPKG        = os.path.join(ROOT_DIR, "Final_ROWs_Delaware.gpkg")
CENTERLINE_GPKG = os.path.join(ROOT_DIR, "Delaware_ROW_centerline.gpkg")
LOG_FILE        = os.path.join(ROOT_DIR, "centerline_log.txt")

# Interpolation distance for Voronoi densification
# Smaller = more detailed centerline but slower
# For a ROW polygon, 5–10m is a good starting point
INTERPOLATION_DISTANCE = 100  # meters

# Simplification tolerance — removes noise/jaggedness
# Higher = smoother but less precise
SIMPLIFY_TOLERANCE = 50  # meters

# =====================================================
# LOGGER — console + file
# =====================================================
fmt = logging.Formatter('[%(asctime)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S')

logger = logging.getLogger()
logger.setLevel(logging.INFO)

console_handler = logging.StreamHandler()
console_handler.setFormatter(fmt)
logger.addHandler(console_handler)

file_handler = logging.FileHandler(LOG_FILE, mode='w')
file_handler.setFormatter(fmt)
logger.addHandler(file_handler)

# =====================================================
# HELPERS
# =====================================================
def eta(start, done, total):
    if done == 0:
        return "estimating..."
    rate = done / (time.time() - start)
    return str(timedelta(seconds=int((total - done) / rate)))

def smooth_line(line, smooth_iterations=3):
    """
    Chaikin's corner cutting algorithm — smooths a LineString
    by iteratively averaging coordinates.
    """
    coords = np.array(line.coords)
    for _ in range(smooth_iterations):
        new_coords = []
        for i in range(len(coords) - 1):
            p1 = coords[i]
            p2 = coords[i + 1]
            q = 0.75 * p1 + 0.25 * p2
            r = 0.25 * p1 + 0.75 * p2
            new_coords.extend([q, r])
        new_coords.append(coords[-1])
        coords = np.array(new_coords)
    return LineString(coords)

start_time = time.time()
logger.info("=" * 60)
logger.info("ROW centerline generation — START")
logger.info("=" * 60)
logger.info(f"  Interpolation distance : {INTERPOLATION_DISTANCE}m")
logger.info(f"  Simplify tolerance     : {SIMPLIFY_TOLERANCE}m")

# =====================================================
# LOAD DATA
# =====================================================
logger.info("Loading ROW polygon...")
rows = gpd.read_file(ROW_GPKG)
logger.info(f"  Loaded {len(rows):,} polygon(s) | CRS: {rows.crs}")

# Ensure projected CRS (meters) — centerline generation requires it
if rows.crs.is_geographic:
    raise ValueError(
        "ROW polygon must be in a projected CRS (meters). "
        "Reproject to something like EPSG:26918 (UTM Zone 18N for Delaware) first."
    )

# =====================================================
# STEP 1 — DISSOLVE TO SINGLE POLYGON
# If already a single dissolved polygon this is a no-op
# =====================================================
logger.info("Dissolving to single polygon...")
dissolved = rows.dissolve().reset_index(drop=True)
logger.info(f"  Dissolved to {len(dissolved):,} polygon(s)")

# =====================================================
# STEP 2 — GENERATE CENTERLINE PER POLYGON
# =====================================================
logger.info("-" * 60)
logger.info("STEP 2: Generating centerlines...")

cl_start    = time.time()
total_polys = len(dissolved)
all_lines   = []

for i, row in enumerate(dissolved.itertuples(), start=1):
    poly = row.geometry

    if poly is None or poly.is_empty:
        logger.warning(f"  Polygon {i} is empty — skipping")
        continue

    logger.info(f"  Processing polygon {i}/{total_polys}...")

    try:
        # Generate Voronoi-based centerline
        cl = Centerline(poly, interpolation_distance=INTERPOLATION_DISTANCE)
        lines = list(cl.geometry.geoms) if cl.geometry.geom_type == "MultiLineString" \
                else [cl.geometry]

        logger.info(f"    Raw centerline segments: {len(lines):,}")

        # Merge connected line segments into fewer, longer lines
        merged = linemerge(unary_union(lines))
        if merged.geom_type == "LineString":
            merged_lines = [merged]
        elif merged.geom_type == "MultiLineString":
            merged_lines = list(merged.geoms)
        else:
            merged_lines = lines

        logger.info(f"    After merge: {len(merged_lines):,} segment(s)")

        # Simplify — removes Voronoi noise/jaggedness
        simplified = [
            line.simplify(SIMPLIFY_TOLERANCE, preserve_topology=True)
            for line in merged_lines
            if not line.is_empty
        ]

        # Smooth — Chaikin corner cutting
        smoothed = [smooth_line(line, smooth_iterations=3) for line in simplified]

        all_lines.extend(smoothed)
        logger.info(f"    Final segments after simplify + smooth: {len(smoothed):,}")

    except Exception as e:
        logger.warning(f"  Failed on polygon {i}: {e}")
        continue

    logger.info(f"  Polygon {i} done | ETA {eta(cl_start, i, total_polys)}")

logger.info(f"  Total centerline segments: {len(all_lines):,}")

# =====================================================
# STEP 3 — BUILD OUTPUT GDF
# =====================================================
logger.info("-" * 60)
logger.info("STEP 3: Building output GeoDataFrame...")

cl_gdf = gpd.GeoDataFrame(
    {"centerline_id": range(len(all_lines))},
    geometry=all_lines,
    crs=rows.crs
)

# Drop any degenerate geometries
cl_gdf = cl_gdf[
    cl_gdf.geometry.is_valid &
    ~cl_gdf.geometry.is_empty
].reset_index(drop=True)

cl_gdf["centerline_id"] = cl_gdf.index
cl_gdf["length_m"]      = cl_gdf.geometry.length

logger.info(f"  Final centerline segments: {len(cl_gdf):,}")
logger.info(f"  Total centerline length  : {cl_gdf['length_m'].sum():,.1f}m")

logger.info(f"  Writing → {CENTERLINE_GPKG}")
cl_gdf.to_file(CENTERLINE_GPKG, driver="GPKG")

# =====================================================
# DONE
# =====================================================
elapsed = timedelta(seconds=int(time.time() - start_time))
logger.info("=" * 60)
logger.info(f"Workflow complete in {elapsed}")
logger.info(f"  Centerline output → {CENTERLINE_GPKG}")
logger.info(f"  Log file          → {LOG_FILE}")
logger.info("=" * 60)